# 05 🧩 部品を足す: generate / 自作top-p / 自作の繰り返しペナルティ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kagioneko/cse-frog/blob/main/notebooks/05_add_parts.ipynb)

> 解説は [HANDBOOK](../docs/HANDBOOK.md) と [CONFIG_GUIDE](../docs/CONFIG_GUIDE.md)、全設定は [REFERENCE](../docs/REFERENCE.md) にあります。

In [1]:
# 🐸 の準備(Colab では最初に1回だけインストールします)
try:
    import cse
except ImportError:
    %pip install -q cse-frog
from cse import Frog, END

## 1. 🐸にしゃべらせる
迷路の道順をいくつか覚えさせて、続きを作らせます。

In [2]:
routes = [list("右右下右下"), list("右下右右下"), list("下右右下右"), list("右右下下右")] * 3
frog = Frog().learn(routes)
print("greedy  :", frog.generate("右", n=8, greedy=True))
for s in range(4):
    print(f"seed={s}  :", frog.generate("右", n=8, seed=s))

greedy  : 下右下右下右下右
seed=0  : 右下右下右下右下
seed=1  : 下右下右下右下右
seed=2  : 右下右下右下右下
seed=3  : 下右下右下右下右


greedy が「下右下右…」と同じ並びをくり返して止まらないのは、🐸が **今の記号と直前2つの並び** しか見ていないからです。どの道順の何歩目かを覚えていないので、ぐるぐる回ります(大きなLLMが遠くの文脈を見るのは、こうならないためでもあります)。

温度を上げると、くじの当たり方がばらけます。高すぎると、すぐ `<END>` を引いて何もしゃべらないこともあります。

In [3]:
for t in [0.2, 0.8, 3.0]:
    f = Frog(temperature=t).learn(routes)
    print(f"temperature={t:<4}", [f.generate("右", n=8, seed=s) for s in range(4)])

temperature=0.2  ['下右下右下右下右', '下右下右下右下右', '下右下右下右下右', '下右下右下右下右']
temperature=0.8  ['右下右下右下右下', '下右下右下右下右', '右下右下右下右下', '下右下右下右下右']
temperature=3.0  ['右右下右下右右下', '下下', '', '下右下']


## 2. 生成の1歩を解剖する
`generate()` は1歩ずつ `probabilities()` を使っているので、途中のどの1歩も `show()` でのぞけます。

In [4]:
out = frog.generate("右", n=8, seed=1)
print("生成:", out)
frog.show("右" + out[:3])      # 4歩目を選んだときの中身

生成: 下右下右下右下右
🐸 ['右', '下', '右', '下'] の次の候補
候補            直接      履歴      並び      痕跡      合計点      確率
右          4.268   0.000   0.500   0.000    4.768   0.960
<END>      1.704   0.000   0.333   0.000    2.037   0.032
下          0.857   0.000   0.167   0.000    1.024   0.009


## 3. 自作サンプラー: top-p
確率の高い順に足して、合計が p を超えたところで切ります。**🐸は先に `top_k_edges` で候補を絞る**ので、効かせるには候補を残しておく必要があります。

In [5]:
import random

def top_p_generate(frog, prefix, n=8, p=0.9, seed=0):
    rng, out = random.Random(seed), []
    for _ in range(n):
        ranked = frog.top(list(prefix) + out, k=100)
        keep, acc = [], 0.0
        for sym, q in ranked:
            keep.append((sym, q))
            acc += q
            if acc >= p:
                break
        r, acc, pick = rng.random() * sum(q for _, q in keep), 0.0, keep[-1][0]
        for sym, q in keep:
            acc += q
            if r < acc:
                pick = sym
                break
        if pick is END:
            break
        out.append(pick)
    return out

data5 = [["A", "B"]] * 5 + [["A", "C"]] * 4 + [["A", "D"]] * 3 + [["A", "E"]] * 2 + [["A", "F"]] * 1
for k in [3, 100]:
    f = Frog(top_k_edges=k).learn(data5)
    firsts = sorted({tuple(top_p_generate(f, ["A"], n=1, p=0.9, seed=s)) for s in range(300)})
    print(f"top_k_edges={k:<4} top-p 0.9 で出てくる最初の記号: {firsts}")

top_k_edges=3    top-p 0.9 で出てくる最初の記号: [('B',), ('C',), ('D',)]


top_k_edges=100  top-p 0.9 で出てくる最初の記号: [('B',), ('C',), ('D',), ('E',)]

`top_k_edges=3` では、top-p を足しても B・C・D しか出ません。その前に3匹に絞られているからです。`top_k_edges=100` にすると E まで出て、F は top-p で切られます。

## 4. 自作の繰り返しペナルティ(禁止 vs 減点)
不応期は最近の記号を **0にする(禁止)**。ここでは **割り算で下げる(減点)** 部品を自作します(HANDBOOK の Experiment 11)。

In [6]:
import math

data = [["右", "右", "下", "右", "左"]] * 3 + [["右", "右", "下", "上", "左"]] * 1
prefix = ["右", "右", "下"]           # 次の正解は 4回中3回「右」(距離2の繰り返し)

def soft_penalty(frog, prefix, theta, window=2):
    recent = set(list(prefix)[-window:])
    sc = {s: (v / theta if s in recent else v) for s, v in frog.scores(prefix).items() if v > 0}
    top = sorted(sc.items(), key=lambda kv: -kv[1])[:frog.config.top_k_edges]
    t = frog.config.temperature
    m = max(v for _, v in top)
    ex = {s: math.exp((v - m) / t) for s, v in top}
    z = sum(ex.values())
    return [(s, round(e / z, 3)) for s, e in sorted(ex.items(), key=lambda kv: -kv[1])]

frog = Frog().learn(data)
print("抑制なし      :", [(s, round(p, 3)) for s, p in frog.top(prefix)])
print("禁止(不応期2) :", [(s, round(p, 3)) for s, p in Frog(refractory_steps=2).learn(data).top(prefix)])
for theta in (1.5, 4, 100):
    print(f"減点 θ={theta:<5}:", soft_penalty(frog, prefix, theta))

抑制なし      : [('右', 0.796), ('上', 0.204)]
禁止(不応期2) : [('上', 1.0)]
減点 θ=1.5  : [('右', 0.663), ('上', 0.337)]
減点 θ=4    : [('上', 0.543), ('右', 0.457)]
減点 θ=100  : [('上', 0.66), ('右', 0.34)]


- θ=4 で、正しい繰り返し(右)が1位から落ちます。
- θ=100 でも右は残ります。**点数がプラスなら候補に残る**ので、割り算では「禁止」になりません。

部品は「足せば効く」のではなく、ほかの候補との点数の関係で効き方が決まります。🐸に足せない部品(アテンション・勾配降下)については、HANDBOOK の「🐸の先へ」へ。🐸